# MIPROv2

Jointly propose instructions and demonstrations, then use validation feedback to search their combinations.

**Use it when:** Both prompt wording and examples may matter and you can spend more compile calls for a joint search.

**What compilation changes:** Instruction plus up to two bootstrapped and two labeled demonstrations.

Important configuration in this benchmark:

- `auto='light'` search budget
- seed 42 with minibatched validation
- Sol proposes prompts; Luna runs the task

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'miprov2'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='miprov2'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.MIPROv2(
    metric=exact_match, prompt_model=reflection_lm, task_model=task_lm,
    auto='light', max_bootstrapped_demos=2, max_labeled_demos=2, seed=42,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, valset=valset, minibatch=True,
    requires_permission_to_run=False,
)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: miprov2
task model: openai/gpt-5.6-luna
final test accuracy: 66.2% (53/80)
optimized validation accuracy: 76.7%
same-model baseline: 53.8%
uplift: +12.5 points
optimization cost: $0.3052
optimization time: 270.8s
mean inference latency: 1.628s
p95 inference latency: 2.874s

Published artifacts:
- program snapshot: chapter06/results/expanded_notebooks/miprov2/full/optimized_program.json
- prompt snapshot: chapter06/results/expanded_notebooks/miprov2/full/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Inspect instruction and demos together: MIPROv2 optimizes their combination, so attributing the result to either one alone would be misleading.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (1308 characters):
You are a forensic editor specializing in distinguishing human-written software documentation from AI-paraphrased technical prose. Given a passage, decide whether it is likely AI-generated or human-authored.

Base the judgment primarily on stylistic evidence rather than technical subject matter. AI paraphrases often sound unusually polished or generalized and may contain explicit transitions, parenthetical explanations, balanced contrasts, elevated or inconsistent diction, dense punctuation, redundant restatements, or formulaic phrases such as “that is,” “as needed,” and “tend to.” Human documentation is often more direct, compact, project-specific, informal, or editorially imperfect. Treat length, formatting, correct terminology, and grammaticality as weak signals because both classes may preserve these features.

Evaluate the passage holistically and comparatively: ask whether its wording resembles an original documenta

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.